# Stage 11 prereq — landmark extraction for the 122-signer paper-split dataset

Walks the 6 paper-split zips (`eng_{train,val,test}_{lex,nonlex}.zip`) and writes a per-clip .npz layout under `/kaggle/working/landmark_cache_122/`.  No training here.

**MediaPipe is pinned** to match the version that produced the 38-signer cache.  A version mismatch silently changes coordinate normalisation; the sanity-check Cell 5 catches that within ±10% feature mean / std.

**Disk discipline** (Kaggle's 20 GB /kaggle/working/ limit + 70 GB /kaggle/temp/):
- Streams from the input zips one at a time via Python's `zipfile` — no full extraction.
- Per-clip output is fp16 .npz (~6 KB each) -> ~250 MB total cache.
- `_DONE` markers per zip survive session disconnects.

**Wall-clock**: ~3 h CPU.  GPU off (MediaPipe is CPU-bound).

## After this kernel commits

Save Version -> Save & Run All.  Then upload `/kaggle/working/landmark_cache_122/` as a new private Kaggle dataset (e.g. `wita-full-english-landmark-cache`).  The Stage 11 training notebook attaches that dataset.

## Cell 1 — Install + clone (MediaPipe pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
# PIN MediaPipe to the same version that produced the 38-signer cache.
# If the sanity check (Cell 5) fails the ±10% drift bound, bump this.
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe; print(f'mediapipe version: {mediapipe.__version__}')

## Cell 2 — Locate the 6 paper-split zips

In [ ]:
import os, glob, re

# ============================================================================
# MANUAL OVERRIDE (use this if auto-detection below fails).
# Map each of the 6 paper-split zips to (split, subset).  Leave as None to
# auto-detect from filename patterns.
# Example:
#   MANUAL_ZIP_MAPPING = {
#       '/kaggle/input/wita-full-english-122signers/train_lex.zip': ('train','lex'),
#       ...
#   }
MANUAL_ZIP_MAPPING = None
# ============================================================================

INPUT_ROOTS = sorted(glob.glob('/kaggle/input/*'))
print('Mounted inputs:')
for r in INPUT_ROOTS:
    print(f'  {r}')

# ----- Diagnostic: list EVERY .zip anywhere under /kaggle/input/ -----
print('\n--- ALL .zip files under /kaggle/input/ (any depth) ---')
all_zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
for p in all_zips:
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {size_mb:>8.1f} MB  {p}')
print(f'\nFound {len(all_zips)} total .zip files.')

# ----- If user provided a manual mapping, use it -----
if MANUAL_ZIP_MAPPING is not None:
    zip_paths = list(MANUAL_ZIP_MAPPING.keys())
    print(f'\nUsing MANUAL_ZIP_MAPPING with {len(zip_paths)} entries.')
else:
    # ----- Permissive auto-detection -----
    # Allow many filename conventions:
    #   eng_train_lex.zip          (original spec)
    #   train_lex.zip / lex_train.zip
    #   english_train_lex.zip / wita_train_lex.zip
    #   With dashes instead of underscores; case-insensitive.
    SPLIT_TOK  = r'(train|valid?|test|tst|tr|va|te)'
    SUBSET_TOK = r'(non[_-]?lex|nonlex|lex|freq[_-]?word|non[_-]?freq[_-]?word)'
    PAT_SPLIT_FIRST  = re.compile(rf'{SPLIT_TOK}[_-]{SUBSET_TOK}',  re.I)
    PAT_SUBSET_FIRST = re.compile(rf'{SUBSET_TOK}[_-]{SPLIT_TOK}', re.I)

    def _classify(name: str):
        n = os.path.basename(name).lower()
        for pat, order in [(PAT_SPLIT_FIRST, 'split_first'),
                           (PAT_SUBSET_FIRST, 'subset_first')]:
            m = pat.search(n)
            if not m:
                continue
            if order == 'split_first':
                split_raw, subset_raw = m.group(1), m.group(2)
            else:
                subset_raw, split_raw = m.group(1), m.group(2)
            # Normalise.
            split_map = {'train':'train','tr':'train',
                         'val':'val','valid':'val','va':'val',
                         'test':'test','tst':'test','te':'test'}
            split = split_map.get(split_raw, split_raw)
            if re.search(r'non', subset_raw):
                subset = 'nonlex'
            elif 'freq' in subset_raw and 'non' not in subset_raw:
                subset = 'lex'      # "freq_word" = lex (paper convention)
            elif subset_raw == 'lex':
                subset = 'lex'
            else:
                subset = None
            if split in ('train','val','test') and subset in ('lex','nonlex'):
                return split, subset
        return None

    zip_paths = []
    classifications = {}
    for p in all_zips:
        c = _classify(p)
        if c is not None:
            zip_paths.append(p)
            classifications[p] = c

    print(f'\nAuto-detected {len(zip_paths)} paper-split zips:')
    for p in zip_paths:
        s, ss = classifications[p]
        print(f'  {os.path.basename(p):<40s} -> {s}/{ss}')

# ----- Confirm we have the expected 6 -----
if len(zip_paths) != 6:
    print('\n!!! Expected 6 zips (train/val/test × lex/nonlex), found '
          f'{len(zip_paths)}. !!!\n'
          'Either:\n'
          '  (a) verify the dataset was attached: see "ALL .zip files" listing above\n'
          '  (b) the filename convention is unusual -- set MANUAL_ZIP_MAPPING at the\n'
          '      top of THIS cell to a dict {zip_path: (split, subset)}, re-run\n')
    # Don't hard-assert here so the diagnostic listing stays visible.
    # The next cell asserts when it tries to use zip_paths.
else:
    print('\n✅ Found 6 paper-split zips. Cell 3 will set up the per-zip resume markers.')

## Cell 3 — Output paths + resume markers

In [ ]:
OUT_ROOT = '/kaggle/working/landmark_cache_122'
os.makedirs(OUT_ROOT, exist_ok=True)
MARKER_DIR = os.path.join(OUT_ROOT, '_markers')
os.makedirs(MARKER_DIR, exist_ok=True)

# Build the canonical zip_path -> (split, subset) dict from whichever path
# the previous cell took (auto-detection or manual override).
if MANUAL_ZIP_MAPPING is not None:
    ZIP_TO_LABEL = dict(MANUAL_ZIP_MAPPING)
else:
    ZIP_TO_LABEL = dict(classifications)

assert len(ZIP_TO_LABEL) == 6, (
    f'Need 6 zips classified (train/val/test × lex/nonlex), got '
    f'{len(ZIP_TO_LABEL)}.  Re-run Cell 2 with MANUAL_ZIP_MAPPING set.'
)
# Sanity-check coverage: every (split, subset) pair appears exactly once.
covered = sorted(ZIP_TO_LABEL.values())
expected = sorted([(s, ss) for s in ('train','val','test')
                          for ss in ('lex','nonlex')])
assert covered == expected, (
    f'Coverage mismatch.\n  expected: {expected}\n  got:      {covered}'
)

def parse_split_subset(zip_path):
    return ZIP_TO_LABEL[zip_path]

def marker_path(zip_path):
    split, subset = parse_split_subset(zip_path)
    return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')

print('Final zip -> (split, subset) mapping:')
for p, (s, ss) in sorted(ZIP_TO_LABEL.items(), key=lambda kv: (kv[1][0], kv[1][1])):
    mk = marker_path(p)
    print(f'  {os.path.basename(p):<40s} -> {s}/{ss}  '
          f'(marker {"exists" if os.path.exists(mk) else "missing"})')

# Stable order for Cell 4: smallest splits first so the cheap rows finish
# before any session-length risk.  This puts val/test before train.
SPLIT_ORDER = {'val': 0, 'test': 1, 'train': 2}
zip_paths = sorted(ZIP_TO_LABEL.keys(),
                   key=lambda p: (SPLIT_ORDER[ZIP_TO_LABEL[p][0]],
                                  ZIP_TO_LABEL[p][1]))

## Cell 4 — Extract per-clip landmarks  (~30 min per zip on Kaggle CPU)

Streams each zip in turn, writing `<SIGNER>__<clip_id>.npz` files.  Resume-aware via `_DONE` markers.

In [ ]:
from wita_v2.datasets.landmark_cache_122 import extract_zip_per_clip_landmarks
from wita_v2.datasets.skeleton_cache        import LandmarkExtractor

extractor = LandmarkExtractor()
all_stats = {}
for p in zip_paths:
    split, subset = parse_split_subset(p)
    mk = marker_path(p)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done')
        continue
    print(f'\n>>> extracting {split}/{subset}  from {p}')
    stats = extract_zip_per_clip_landmarks(
        zip_path  = p,
        out_dir   = OUT_ROOT,
        split     = split,
        subset    = subset,
        lang      = 'english',
        max_frames= 64,
        T_native  = 32,
        extractor = extractor,        # reuse a single MediaPipe instance
        overwrite = False,
    )
    all_stats[f'{split}_{subset}'] = stats
    with open(mk, 'w') as f:
        import json; json.dump(stats, f, indent=2, default=str)
extractor.close()
print('\nAll zips processed.')

## Cell 5 — Sanity check: feature shape + value range

In [ ]:
import numpy as np
from pathlib import Path
import random

all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print(f'Total .npz files: {len(all_npz)}')
assert len(all_npz) > 0, 'No clips extracted'

random.seed(42)
sample = random.sample(all_npz, min(100, len(all_npz)))
feats = np.stack([np.load(p, allow_pickle=False)['feature'].astype(np.float32) for p in sample])
print(f'feature shape per clip : {feats.shape[1:]}')
print(f'feature dtype          : {feats.dtype}')
print(f'feature mean           : {feats.mean():.4f}')
print(f'feature std            : {feats.std():.4f}')
print(f'feature min/max        : {feats.min():.4f} / {feats.max():.4f}')
assert feats.shape[1:] == (32, 190), f'Bad shape: {feats.shape[1:]}'
assert np.all(np.isfinite(feats)), 'Non-finite values present'

# Per-split counts.
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        n = len(list((Path(OUT_ROOT) / split / subset).glob('*.npz')))
        print(f'  {split}/{subset:<7s}: {n}')

## Cell 6 — Commit kernel + next step

1. **Save Version -> Save & Run All**.  The committed kernel's output dataset contains `landmark_cache_122/`.
2. After it commits, go to **Datasets -> New Dataset -> Notebook Output**, name it `wita-full-english-landmark-cache`.
3. Attach that dataset to the Stage 11 training notebook (next kernel).